In [ ]:
import warnings
warnings.filterwarnings("ignore")


In [ ]:
!pip install pandas


In [ ]:
import pandas as pd
import openpyxl

In [ ]:
df_gpt = pd.read_csv('/content/GPT_Results.csv')
df_gpt.head()

In [ ]:
df_comments = pd.read_excel('/content/positive_extracts_random_sample_comments.xlsx')
df_comments.head()

In [ ]:
!pip install readability


# New Section

In [ ]:
import nltk
nltk.download('punkt')


In [ ]:
#from py_readability_metrics import *

In [ ]:
#from py_readability_metrics.readability.readability import *

In [ ]:
#r = readability.readability.Readability(df_gpt.loc[0, 'response'])

In [ ]:
!pip install evaluate


In [ ]:
!pip install rouge-score


In [ ]:
# Usage of BLEU and ROUGE scores .
import transformers
import evaluate
from rouge_score import rouge_scorer, scoring
# For usage documentation: https://huggingface.co/spaces/evaluate-metric/rouge
rouge = evaluate.load('rouge')

In [ ]:
# For usage documentation: https://huggingface.co/spaces/evaluate-metric/bleu
bleu = evaluate.load("bleu")

In [ ]:
!unzip /content/readability.zip

In [ ]:
from readability import *

In [ ]:
from readability import Readability


In [ ]:
#r = Readability(df_comments.loc[0, 'comment_content'])

In [ ]:
df_comments.shape

In [ ]:
result = pd.DataFrame(columns=['row_id','ari_score','ari_gl','ari_arg','smog_score','smog_gl'])

In [ ]:
from readability.exceptions import ReadabilityException


In [ ]:
for i in range(0, 625):
    r = Readability(df_comments.loc[i, 'comment_content'])
    print(i)
    if len(df_comments.loc[i, 'comment_content'].split()) < 100:
        continue
    try:
        ari = r.ari()
        s = r.smog(all_sentences=True, ignore_length=True)
        result = result.append({'row_id': i, 'ari_score': ari.score,
                                'ari_gl': ari.grade_levels, 'ari_arg': ari.ages,
                                'smog_score': s.score, 'smog_gl': s.grade_level}, ignore_index=True)
    except ReadabilityException as e:
        print(f"Skipping row {i} due to: {e}")


In [ ]:
result

In [ ]:
import pandas as pd

result = pd.DataFrame(columns=['row_id', 'question_post_id', 'comment_post_id', 'ari_score', 'ari_gl', 'ari_arg', 'smog_score', 'smog_gl'])


In [ ]:
for i in range(0, 625):
    r = Readability(df_comments.loc[i, 'comment_content'])
    print(i)
    if len(df_comments.loc[i, 'comment_content'].split()) < 100:
        continue
    try:
        ari = r.ari()
        s = r.smog(all_sentences=True, ignore_length=True)
        result = result.append({
            'row_id': i,
            'question_post_id': df_comments.loc[i, 'question_post_id'],
            'comment_post_id': df_comments.loc[i, 'comment_post_id'],
            'ari_score': ari.score,
            'ari_gl': ari.grade_levels,
            'ari_arg': ari.ages,
            'smog_score': s.score,
            'smog_gl': s.grade_level,
        }, ignore_index=True)
    except Exception as e:
        print(f"Skipping row {i} due to: {e}")


In [ ]:
result

In [ ]:
result.to_csv('/content/results_coomments.csv', index=False)

In [ ]:
df_comments2 = pd.read_csv('/content/results_coomments (1).csv')
df_comments2.head()

In [ ]:
df_comments2.shape

In [ ]:
import pandas as pd
from readability import Readability

average_scores_comments = result.groupby('question_post_id')[['ari_score', 'smog_score']].mean().reset_index()
average_scores_comments.columns = ['question_post_id', 'avg_ari_comments', 'avg_smog_comments']


def calculate_scores(text):
    try:
        r = Readability(text)
        ari = r.ari()
        s = r.smog(all_sentences=True, ignore_length=True)
        return ari.score, s.score
    except Exception as e:
        return None, None


df_gpt['ari_response'], df_gpt['smog_response'] = zip(*df_gpt['response'].apply(calculate_scores))
df_gpt['ari_original'], df_gpt['smog_original'] = zip(*df_gpt['question_content/Original Post'].apply(calculate_scores))


df_gpt = pd.merge(df_gpt, average_scores_comments, on='question_post_id', how='left')

print(df_gpt[['question_post_id', 'ari_response', 'smog_response', 'ari_original', 'smog_original', 'avg_ari_comments', 'avg_smog_comments']].head())


In [ ]:
df_gpt.to_csv('/content/results_coomments_average.csv', index=False)

In [ ]:
df_gpt_cleaned = df_gpt.dropna(subset=['avg_ari_comments', 'avg_smog_comments'])
df_gpt_cleaned.to_csv('/content/cleaned_results.csv', index=False)


In [ ]:
non_null_rows_count = df_gpt[['avg_ari_comments', 'avg_smog_comments']].notnull().all(axis=1).sum()

print(f"Number of rows with non-null 'avg_ari_comments' and 'avg_smog_comments': {non_null_rows_count}")
